In [1]:
import pandas as pd
import numpy as np
import os # 导入 os 库来创建文件夹

# --- [步骤 1] 定义文件路径 ---

input_file = '../data/raw/kaggle_data/PRSA_Data_Aotizhongxin_20130301-20170228.csv'
output_dir = '../data/processed'
output_file = os.path.join(output_dir, 'Aotizhongxin_Processed.csv')

# ----------------------------------------------------

print(f"原始文件: {input_file}")
print(f"目标文件: {output_file}")

try:
    # --- [步骤 2] 加载原始数据 ---
    df = pd.read_csv(input_file)
    print(f"\n--- [步骤 2] 成功加载原始数据，共 {len(df)} 行 ---")

    # --- [步骤 3] 创建 'datetime' 列 ---
    # 将 'year', 'month', 'day', 'hour' 合并为 pandas 可识别的 datetime 对象
    df['datetime'] = pd.to_datetime(df[['year', 'month', 'day', 'hour']])
    print("--- [步骤 3] 'datetime' 列创建成功 ---")

    # --- [步骤 4] 处理缺失值 (NaN) ---
    print("--- [步骤 4] 开始处理缺失值 (NaN)... ---")
    
    # 关键：处理目标变量 'PM2.5'
    # 我们使用 'linear' (线性插值) 来填充空洞
    df['PM2.5'] = df['PM2.5'].interpolate(method='linear')
    # 如果文件最开头就有 NaN，插值会失败，我们用 'bfill' (向后填充) 来补上
    df['PM2.5'] = df['PM2.5'].bfill()
    print(" > 'PM2.5' 缺失值处理完毕。")

    # 处理其他所有数值型气象特征
    weather_cols = ['PM10', 'SO2', 'NO2', 'CO', 'O3', 'TEMP', 'PRES', 'DEWP', 'RAIN', 'WSPM']
    for col in weather_cols:
        if col in df.columns:
            # 同样使用线性插值，然后用 0 填充开头或结尾的剩余 NaN
            df[col] = df[col].interpolate(method='linear').fillna(0)
    print(" > 其他气象特征缺失值处理完毕。")

    # --- [步骤 5] 设置 'datetime' 为索引 ---
    # 这是时间序列分析的标准步骤，便于后续按时间切片和绘图
    df.set_index('datetime', inplace=True)
    print("--- [步骤 5] 'datetime' 已设置为索引 ---")
    
    # --- [新步骤 5b] 填充 'wd' (风向) 的缺失值 ---
    # 我们使用 'ffill' (forward fill)
    # 这会用上一个有效的小时的风向来填充当前的 NaN
    df['wd'] = df['wd'].fillna(method='ffill')
    # 再用 'bfill' (backward fill) 填充掉文件最开头的 NaN
    df['wd'] = df['wd'].bfill()
    print("--- [步骤 5b] 'wd' 列的缺失值已填充 ---")

    # --- [步骤 6] 转换文本特征 'wd' (风向) ---
    # 使用 "独热编码" (One-Hot Encoding) 将文本转换为 0 和 1
    df = pd.get_dummies(df, columns=['wd'], prefix='wind')
    print("--- [步骤 6] 'wd' (风向) 已转换为独热编码 ---")

    # --- [步骤 7] 清理不再需要的列 ---
    # 'No' 是行号, 'year', 'month', 'day', 'hour', 'station' 已被 'datetime' 索引和文件名替代
    cols_to_drop = ['No', 'year', 'month', 'day', 'hour', 'station']
    df_processed = df.drop(columns=cols_to_drop, errors='ignore')
    print("--- [步骤 7] 原始列清理完毕 ---")

    # --- [步骤 8] 保存处理好的文件 ---
    # 确保 'data/processed' 文件夹存在
    os.makedirs(output_dir, exist_ok=True)
    
    # 将我们干净的 DataFrame 保存到新的 CSV 文件中
    df_processed.to_csv(output_file)
    print(f"\n--- [步骤 8] 成功！处理后的文件已保存到: {output_file} ---")

    # --- [最终] 预览最终成果 ---
    print("\n处理后数据的 (前5行) 预览：")
    print(df_processed.head())


except FileNotFoundError:
    print(f"\n[!! 错误 !!] 找不到文件: {input_file}")
    print("请确认您的 .ipynb 文件保存在 'Project' 文件夹中，")
    print("并且数据文件在 'data/kaggle_data/' 文件夹下。")
except Exception as e:
    print(f"\n[!! 错误 !!] 处理过程中发生意外错误: {e}")

原始文件: ../data/raw/kaggle_data/PRSA_Data_Aotizhongxin_20130301-20170228.csv
目标文件: ../data/processed\Aotizhongxin_Processed.csv

--- [步骤 2] 成功加载原始数据，共 35064 行 ---
--- [步骤 3] 'datetime' 列创建成功 ---
--- [步骤 4] 开始处理缺失值 (NaN)... ---
 > 'PM2.5' 缺失值处理完毕。
 > 其他气象特征缺失值处理完毕。
--- [步骤 5] 'datetime' 已设置为索引 ---
--- [步骤 5b] 'wd' 列的缺失值已填充 ---
--- [步骤 6] 'wd' (风向) 已转换为独热编码 ---
--- [步骤 7] 原始列清理完毕 ---

--- [步骤 8] 成功！处理后的文件已保存到: ../data/processed\Aotizhongxin_Processed.csv ---

处理后数据的 (前5行) 预览：
                     PM2.5  PM10   SO2   NO2     CO    O3  TEMP    PRES  DEWP  \
datetime                                                                        
2013-03-01 00:00:00    4.0   4.0   4.0   7.0  300.0  77.0  -0.7  1023.0 -18.8   
2013-03-01 01:00:00    8.0   8.0   4.0   7.0  300.0  77.0  -1.1  1023.2 -18.2   
2013-03-01 02:00:00    7.0   7.0   5.0  10.0  300.0  73.0  -1.1  1023.5 -18.2   
2013-03-01 03:00:00    6.0   6.0  11.0  11.0  300.0  72.0  -1.4  1024.5 -19.4   
2013-03-01 04:00:00    3.0   3.0  12.

C:\Users\胡杨\AppData\Local\Temp\ipykernel_18852\3873494026.py:52: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['wd'] = df['wd'].fillna(method='ffill')
